In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx

pa.settings.strict_units = False

import pico
halonums = pico.halonums
pa.gpu_init()

Your machine only has 1 available threads


In [2]:
def extract_turbulent_scalar(snap, sf, variable, filter_length, weight, filter_type):
    
    filt_var = sf.filter_variable(variable, filter_length, weight=weight, 
                                      filter_type=filter_type, iterative=True, optimized=True)

    print("min/max/avg occupancy cartesian tiling %d / %d / %.2f"%(sf.tile.particles_per_tile.min(),
                                                                    sf.tile.particles_per_tile.max(),
                                                                    cp.mean(sf.tile.particles_per_tile)))
    
    smoothVar = snap[variable].copy
    smoothVar[sf.index] = filt_var
    
    turbVar = np.zeros_like(snap[variable])
    turbVar[sf.index] = snap[variable][sf.index] - filt_var

    return smoothVar, turbVar


def extract_turbulent_energy(snap, sf, vector_variable, filter_length, weight, filter_type):
    if isinstance(vector_variable, str):
        variable_x = snap[vector_variable][:,0]
        variable_y = snap[vector_variable][:,1]
        variable_z = snap[vector_variable][:,2]
    else:
        variable_x = vector_variable[:,0]
        variable_y = vector_variable[:,1]
        variable_z = vector_variable[:,2]
    
    filt_var_x = sf.filter_variable(variable_x, filter_length, weight=weight, 
                                      filter_type=filter_type, iterative=True, optimized=True)
    filt_var_y = sf.filter_variable(variable_y, filter_length, weight=weight, 
                                      filter_type=filter_type, iterative=True, optimized=True)
    filt_var_z = sf.filter_variable(variable_z, filter_length, weight=weight, 
                                      filter_type=filter_type, iterative=True, optimized=True)

    print("min/max/avg occupancy cartesian tiling %d / %d / %.2f"%(sf.tile.particles_per_tile.min(),
                                                                    sf.tile.particles_per_tile.max(),
                                                                    cp.mean(sf.tile.particles_per_tile)))

    final_filter_length = 0.0*filter_length.copy
    final_filter_length[sf.index] = sf.filter_lengths_out.get()*filter_length.unit

    if isinstance(vector_variable, str):
        smoothEner = np.zeros_like(snap[vector_variable][:,0]**2)
        turbEner = np.zeros_like(snap[vector_variable][:,0]**2)
    else:
        smoothEner = np.zeros_like(vector_variable[:,0]**2)
        turbEner = np.zeros_like(vector_variable[:,0]**2)
    smoothEner[sf.index] = 0.5*(filt_var_x**2 + filt_var_y**2 + filt_var_z**2)
    
    delta_var_x = variable_x[sf.index] - filt_var_x
    delta_var_y = variable_y[sf.index] - filt_var_y
    delta_var_z = variable_z[sf.index] - filt_var_z

    turbEner[sf.index] = 0.5*(delta_var_x**2 + delta_var_y**2 + delta_var_z**2)

    return smoothEner, turbEner, final_filter_length

In [ ]:

# def make_filtered_snapshot(halo):
halo = halonums[0]
for halo in halonums:
    print(f'\n\nWorking on halo {halo}')
    simfolder = pico.get_simfolder(halo)
    info = pico.PicoOutputInfo(f'{simfolder}output/pico_output_info.txt')
    tracking = np.loadtxt(f'{simfolder}output/pico_tracking_info.txt', dtype=int)
    
    snapnum = info.num_vec_full[::-1][0]
    snap = pa.Snapshot(simfolder + 'output/', snapnum, basename='snapshot')
    center = snap.Cat.Group['GroupPos'][0]
    R200 = snap.Cat.Group['Group_R_Crit200'][0]
    widths = 0.5 * np.ones(3) * R200
    
    filter_length = 2.0*snap['0_Diameters']
    weight='0_Volume'
    filter_type = 'gaussian'
    
    # when using the iterative filter we need to make sure that the particles required are
    # loaded on the GPU. This can be achieved by increasing the search_radius
    # to be larger than the maximum filter_radius of the iterative loop
    sf = tc.SmoothingFilter(snap, center, widths, npix=128, orientation=None, 
                            search_radius=5.*filter_length.value)
    
    smoothVar, turbVar = extract_turbulent_scalar(snap, sf, '0_Density', filter_length=filter_length, 
                                                  weight=weight, filter_type=filter_type)
    analysis_folder = pico.get_analysisfolder(halo) + 'turbulence/'
    writer = pa.PaicosWriter(snap, basedir=analysis_folder, basename='filtered')
    # for key in ['TurbDensity', '0_sf_index']:
    writer.write_data('TurbDensity', turbVar)
    writer.write_data('smoothDensity', smoothVar)
    writer.write_data('sf_index', sf.index)
    density_filter_length = sf.filter_lengths_out.get()*filter_length.unit
    writer.write_data('density_filter_length', density_filter_length)
    

    if False:
        smoothspecificEner, turbspecificEner, final_filter_length = extract_turbulent_energy(snap, sf, '0_Velocities', filter_length=filter_length, 
                                                      weight=weight, filter_type=filter_type)
        writer.write_data('turbspecificEner', turbspecificEner)
        writer.write_data('smoothspecificEner', smoothspecificEner)
        specific_kinetic_filter_length = sf.filter_lengths_out.get()*filter_length.unit
        writer.write_data('specific_kinetic_filter_length', specific_kinetic_filter_length)
    else:
        var = np.sqrt(snap['0_Density'])[:, None] * snap['0_Velocities']
        smoothEner, turbEner, final_filter_length = extract_turbulent_energy(snap, sf, var, filter_length=filter_length, 
                                                      weight=weight, filter_type=filter_type)
        writer.write_data('turbEner', turbEner)
        writer.write_data('smoothEner', smoothEner)
        kinetic_filter_length = sf.filter_lengths_out.get()*filter_length.unit
        writer.write_data('kinetic_filter_length', kinetic_filter_length)
    
    
    smoothMagEner, turbMagEner, final_filter_length = extract_turbulent_energy(snap, sf, '0_MagneticField', filter_length=filter_length, 
                                                  weight=weight, filter_type=filter_type)
    
    writer.write_data('turbMagEner', turbMagEner)
    writer.write_data('smoothMagEner', smoothMagEner)
    magnetic_filter_length = sf.filter_lengths_out.get()*filter_length.unit
    writer.write_data('magnetic_filter_length', magnetic_filter_length)
    
    
    writer.finalize()
    sf.release_gpu_memory()
    del snap
    import gc
    gc.collect()



Working on halo 0
99.07 percent of particles (7869489 / 7942980) has converged
min/max/avg occupancy cartesian tiling 0 / 144 / 5.72
97.84 percent of particles (7771605 / 7942980) has converged
97.44 percent of particles (7740036 / 7942980) has converged
98.20 percent of particles (7799859 / 7942980) has converged
min/max/avg occupancy cartesian tiling 0 / 144 / 5.72
99.65 percent of particles (7915045 / 7942980) has converged
99.65 percent of particles (7915424 / 7942980) has converged
99.65 percent of particles (7915218 / 7942980) has converged
min/max/avg occupancy cartesian tiling 0 / 144 / 5.72


Working on halo 1
99.21 percent of particles (8771239 / 8840956) has converged
min/max/avg occupancy cartesian tiling 0 / 212 / 8.21
97.92 percent of particles (8656924 / 8840956) has converged
97.76 percent of particles (8642624 / 8840956) has converged
97.56 percent of particles (8625675 / 8840956) has converged
min/max/avg occupancy cartesian tiling 0 / 212 / 8.21
99.66 percent of pa

In [ ]:
%%bash
nvidia-smi

In [ ]:
turb = pa.PaicosReader(analysis_folder, snapnum, basename='filtered')

In [ ]:
analysis_folder